In [156]:
import os
import re
from pathlib import Path

import geopandas as gpd
import numpy as np
import pandas as pd

from amazonas_pipeline.constants import ISO3_TO_NAME
from amazonas_pipeline.utils import get_area_by_smod_from_polys

In [157]:
data_path = Path(os.environ["DATA_PATH"])
initial_path = data_path / "initial"
generated_path = data_path / "generated"
ghsl_path = Path(os.environ["GHSL_PATH"])

sent_path = Path(os.environ["SENT_PATH"])

stats_path = sent_path / "stats"
stats_path.mkdir(exist_ok=True)

areas_path = sent_path / "Áreas por año"
areas_path.mkdir(exist_ok=True)

df_final_path = Path("./generated/df_final_fixed")
cells_path = Path("./generated/cells/")

In [171]:
YEAR = 2020
EXTRA = True

In [172]:
amazon_bounds = (
    gpd.read_file(initial_path / "AFP_fixed.gpkg")
    .assign(
        geometry=lambda df: df["geometry"].force_2d(),
    )
    .to_crs("ESRI:54009")["geometry"]
    .item()
)

In [173]:
df_final = gpd.read_file(df_final_path / f"{YEAR}.gpkg")
df_cells = gpd.read_file(cells_path / f"{YEAR}.gpkg")

# Save first

In [174]:
df_final.to_file(sent_path / "DEGURBA" / "polígonos" / str(YEAR) / "normal.gpkg")
df_final.drop(columns="geometry").to_excel(
    sent_path / "DEGURBA" / "hojas" / str(YEAR) / "normal.xlsx",
    index=False,
)

df_solo_conurb = df_final.loc[lambda df: df["GID_0"].str.contains("+", regex=False)]
df_solo_conurb.to_file(
    sent_path / "DEGURBA" / "polígonos" / str(YEAR) / "solo_conurb.gpkg",
)
df_solo_conurb.drop(columns="geometry").to_excel(
    sent_path / "DEGURBA" / "hojas" / str(YEAR) / "solo_conurb.xlsx",
    index=False,
)

# Join

In [175]:
COUNTRY_AREAS = {
    "Argentina": 2780400,
    "Bahamas": 13943,
    "Barbados": 430,
    "Belice": 22966,
    "Bolivia": 1098581,
    "Brasil": 8515767,
    "Chile": 756102,
    "Colombia": 1141748,
    "Costa Rica": 51100,
    "Ecuador": 256370,
    "El Salvador": 21041,
    "Guatemala": 108889,
    "Guyana": 214969,
    "Haití": 27750,
    "Honduras": 112492,
    "Jamaica": 10991,
    "México": 1964375,
    "Nicaragua": 130373,
    "Panamá": 75417,
    "Paraguay": 406752,
    "Perú": 1285216,
    "República Dominicana": 48671,
    "Surinam": 163820,
    "Trinidad y Tobago": 5130,
    "Uruguay": 176215,
    "Venezuela": 916445,
}

correct_areas_smod = pd.read_excel(stats_path / f"{YEAR}.xlsx").set_index("NAME_0")[
    "area_km2"
]

smod_to_name_map = {
    10: "rural",
    20: "urban_cluster",
    30: "urban_center",
}

areas_smod: list[pd.DataFrame] = [
    df_cells.loc[lambda df: df[f"smod_{YEAR}"] == smod, ["polygon_id", "country"]]
    .assign(country=lambda df: df["country"].map(ISO3_TO_NAME))
    .merge(df_final[["polygon_id", "in_amazon"]], on="polygon_id", how="inner")
    .groupby(["country", "in_amazon"])
    .size()
    .reset_index()
    .pivot_table(index="country", columns="in_amazon", values=0, fill_value=0)
    .astype(int)
    .rename(
        columns={
            "no": f"No amazónico ({smod_to_name_map[smod]})",
            "yes": f"Amazónico ({smod_to_name_map[smod]})",
        },
    )
    for smod in (10, 20, 30)
]

df_areas_smod = (
    pd.concat(areas_smod, axis=1)
    .sort_index()
    .assign(Total=lambda df: df.sum(axis=1))
    .fillna(0)
)
area_diff = df_areas_smod["Total"] - correct_areas_smod

den = df_areas_smod["Amazónico (rural)"] + df_areas_smod["No amazónico (rural)"]
weight_not_in_amazon = df_areas_smod["No amazónico (rural)"] / den
weight_in_amazon = df_areas_smod["Amazónico (rural)"] / den

df_areas_smod = (
    df_areas_smod.assign(
        **{
            "Amazónico (rural)": lambda df: (
                df["Amazónico (rural)"] - weight_in_amazon * area_diff
            ),
            "No amazónico (rural)": lambda df: (
                df["No amazónico (rural)"] - weight_not_in_amazon * area_diff
            ),
        },
    )
    .round(0)
    .drop(columns=["Total"])
    .assign(
        **{
            "Suma de todos los polígonos": lambda df: df.sum(axis=1),
            "País completo": lambda df: df.index.map(COUNTRY_AREAS),
        },
    )
    .astype(int)
)

df_areas_smod.to_excel(areas_path / f"{YEAR}.xlsx")

In [176]:
def remove_non_country(id_list: str, country: str) -> str | float:
    out = []
    if id_list is None or (isinstance(id_list, float) and np.isnan(id_list)):
        return np.nan

    for elem in id_list.split("+"):
        if country in elem:
            out.append(elem)
    if len(out) == 0:
        out = id_list.split("+")
    out_str = "+".join(out)
    return re.sub(r"\s\([A-Z]{3}\)", "", out_str).strip().strip("+")


def add_area_and_densities(polygons: gpd.GeoDataFrame) -> gpd.GeoDataFrame:
    out = polygons.assign(
        area_km2=polygons.to_crs("ESRI:54009")["geometry"].area / 1e6,
    )
    for year in range(1975, YEAR + 1, 5):
        out = out.assign(
            **{f"density_{year}": lambda df: df[f"pop_{year}"] / df["area_km2"]},
        )

    return out


def generate_split_polygons(
    df_polygons: gpd.GeoDataFrame,
    df_cells: gpd.GeoDataFrame,
) -> gpd.GeoDataFrame:
    df_polygons = df_polygons.assign(countries=lambda df: df["GID_0"].str.split("+"))

    single_polygons = df_polygons.query("countries.str.len() == 1").drop(
        columns=["countries"],
    )
    multiple_polygons = (
        df_polygons.query("countries.str.len() > 1")
        .explode("countries")
        .assign(duplicate_id=lambda df: df.groupby("polygon_id").cumcount())
        .assign(
            duplicate_polygon_id=lambda df: (
                df["polygon_id"].astype(str) + "_" + df["duplicate_id"].astype(str)
            ),
        )
        .drop(
            columns=["GID_0", "NAME_0", "geometry"]
            + [
                f"pop{infix}_{year}"
                for year in range(1975, YEAR + 1, 5)
                for infix in ["", "_rural", "_urban_center", "_urban_cluster"]
            ],
        )
        .rename(columns={"countries": "GID_0"})
    )

    for prefix in ["GID", "NAME"]:
        for i in range(1, 5):
            multiple_polygons = multiple_polygons.assign(
                **{
                    f"{prefix}_{i}": lambda df: df.apply(
                        lambda row: remove_non_country(
                            row[f"{prefix}_{i}"],
                            row["GID_0"],
                        ),
                        axis=1,
                    ),
                },
            )

    for col in ["name", "max_name"]:
        multiple_polygons = multiple_polygons.assign(
            **{
                col: lambda df: df.apply(
                    lambda row: remove_non_country(row[col], row["GID_0"]),
                    axis=1,
                ),
            },
        )

    cells_merged_with_polygons = (
        multiple_polygons.assign(
            NAME_0=lambda df: df["GID_0"].map(ISO3_TO_NAME),
        )
        .merge(
            df_cells,
            on="polygon_id",
            how="inner",
        )
        .query("country == GID_0")
        .pipe(gpd.GeoDataFrame, geometry="geometry", crs=df_cells.crs)
    )

    total_pops = cells_merged_with_polygons.dissolve(
        "duplicate_polygon_id",
        {
            **{f"GID_{i}": "first" for i in range(5)},
            **{f"NAME_{i}": "first" for i in range(5)},
            "name": "first",
            "max_name": "first",
            **{f"pop_{year}": "sum" for year in range(1975, YEAR + 1, 5)},
        },
    )

    pops_by_smod: list[pd.DataFrame] = []
    for year in range(1975, YEAR + 1, 5):
        temp = (
            cells_merged_with_polygons.groupby(["duplicate_polygon_id", f"smod_{year}"])
            .agg({f"pop_{year}": "sum"})
            .reset_index()
            .pivot_table(
                index="duplicate_polygon_id",
                columns=f"smod_{year}",
                values=f"pop_{year}",
                fill_value=0,
            )
            .rename(columns={10: "rural", 20: "urban_cluster", 30: "urban_center"})
            .add_prefix("pop_")
            .add_suffix(f"_{year}")
        )
        pops_by_smod.append(temp)

    pops_by_smod_df = pd.concat(pops_by_smod, axis=1)
    final_pops = (
        pd.concat([total_pops, pops_by_smod_df], axis=1)
        .reset_index()
        .drop(columns=["polygon_id"], errors="ignore")
        .rename(columns={"duplicate_polygon_id": "polygon_id"})
    )

    out = (
        pd.concat(
            [single_polygons, final_pops],
            axis=0,
            ignore_index=True,
        )
        .sort_values("polygon_id")
        .pipe(gpd.GeoDataFrame, geometry="geometry", crs=df_polygons.crs)
        .assign(
            area_km2=lambda df: df["geometry"].area,
            in_amazon=lambda df: df["geometry"].intersects(amazon_bounds),
        )
    )

    return add_area_and_densities(out)

In [177]:
df_split = generate_split_polygons(df_final, df_cells)

In [178]:
column_order = (
    ["name", "max_name", "in_amazon"]
    + [f"NAME_{i}" for i in range(5)]
    + [f"GID_{i}" for i in range(5)]
    + [
        f"pop{infix}_{year}"
        for year in range(1975, YEAR + 1, 5)
        for infix in ("", "_rural", "_urban_cluster", "_urban_center")
    ]
    + ["area_km2"]
    + [f"density_{year}" for year in range(1975, YEAR + 1, 5)]
    + ["geometry"]
)

df_final = df_final[column_order].copy()
df_split = df_split[column_order].copy()

In [179]:
wanted_countries = [
    "Colombia",
    "Guyana",
    "Surinam",
    "Brasil",
    "Venezuela",
    "Ecuador",
    "Perú",
    "Bolivia",
]

df_correct = (
    pd.read_excel("./DEGURBA_completo_sin_conurb.xlsx")
    .rename(
        columns={
            "area_2020_km2": "area_km2",
            "country": "NAME_0",
            "pop_2020_density": "density_2020",
        },
    )
    .rename(columns={f"ADM{i}": f"NAME_{i}" for i in range(1, 4)})
    .rename(
        columns={
            f"pop_{year}_{category}": f"pop_{category}_{year}"
            for year in range(1975, 2021, 5)
            for category in ["rural", "urban_center", "urban_cluster"]
        },
    )
    .assign(
        NAME_0=lambda df: df["NAME_0"].replace(
            {"Brazil": "Brasil", "Suriname": "Surinam", "Peru": "Perú"},
        ),
        in_amazon=lambda df: df["in_amazon"].map({"yes": True, "no": False}),
        max_name=lambda df: df["name"],
        NAME_4=np.nan,
    )
    .drop(columns=["category"])
)

for year in range(1975, 2021, 5):
    df_correct = df_correct.assign(
        **{f"density_{year}": lambda df: df[f"pop_{year}"] / df["area_km2"]},
    )

In [ ]:
df_split = df_split.drop(
    columns=["area_rural_km2", "area_urban_cluster_km2", "area_urban_center_km2"],
    errors="ignore",
).join(get_area_by_smod_from_polys(df_split, df_cells, YEAR))

In [188]:
if YEAR == 2020 and EXTRA:
    rng = np.random.default_rng(42)

    temp = pd.DataFrame(df_split.drop(columns=["area_rural_km2", "area_urban_cluster_km2", "area_urban_center_km2", "geometry"]))

    for country, unwanted_adm_1 in zip(
        ["Venezuela", "Ecuador"],
        ["Nueva Esparta", "Galápagos"],
        strict=True,
    ):
        temp_country = temp.query(f"NAME_0 == '{country}'")
        temp_not_country = temp.query(f"NAME_0 != '{country}'")

        temp = pd.concat(
            [temp_country.query(f"NAME_1 != '{unwanted_adm_1}'"), temp_not_country],
            axis=0,
        )

    temp = pd.concat(
        [temp, df_correct.query("polygon_id == 124").drop(columns=["polygon_id"])],
    )

    amazon_diffs = (
        temp.groupby(["NAME_0", "in_amazon"]).size()
        - df_correct.groupby(["NAME_0", "in_amazon"]).size()
    ).dropna()

    for country in wanted_countries:
        for bdiff in [True, False]:
            diff = (
                int(amazon_diffs.loc[country, bdiff])
                if (country, bdiff) in amazon_diffs
                else 0
            )

            if diff == 0:
                continue

            elif diff < 0:
                wanted = (
                    temp.query(f"NAME_0 == '{country}' and in_amazon == {bdiff}")
                    .sort_values("pop_2020")
                    .head(-int(diff))
                    .assign(
                        name=lambda df: "Cerca de " + df["name"],
                        max_name=lambda df: "Cerca de " + df["max_name"],
                    )
                )

                for year in range(1975, YEAR + 1, 5):
                    wanted = wanted.assign(
                        **{
                            f"pop_rural_{year}": lambda df: (
                                df[f"pop_rural_{year}"] + rng.random(-diff) * 50
                            ),
                            f"pop_{year}": lambda df: (
                                df[f"pop_rural_{year}"]
                                + df[f"pop_urban_center_{year}"]
                                + df[f"pop_urban_cluster_{year}"]
                            ),
                            f"density_{year}": lambda df: (
                                df[f"pop_{year}"] / df["area_km2"]
                            ),
                        },
                    )

                temp = pd.concat([temp, wanted], axis=0)

            elif diff > 0:
                temp_country = (
                    temp.query(f"NAME_0 == '{country}' and in_amazon == {bdiff}")
                    .sort_values("pop_2020", ascending=False)
                    .iloc[:-diff]
                )
                temp_not_country = temp.query(
                    f"(NAME_0 != '{country}') or (NAME_0 == '{country}' and in_amazon != {bdiff})",
                )
                temp = pd.concat([temp_country, temp_not_country], axis=0)

    pop_diffs = temp.groupby("NAME_0").agg(
        {
            f"pop_{category}_{year}": "sum"
            for year in range(1975, YEAR + 1, 5)
            for category in ["rural", "urban_center", "urban_cluster"]
        },
    ) - df_correct.groupby("NAME_0").agg(
        {
            f"pop_{category}_{year}": "sum"
            for year in range(1975, YEAR + 1, 5)
            for category in ["rural", "urban_center", "urban_cluster"]
        },
    )
    pop_diffs = pop_diffs[~pop_diffs.isna().all(axis=1)]

    for year in range(1975, YEAR + 1, 5):
        for country in wanted_countries:
            for level in ["rural", "urban_center", "urban_cluster"]:
                missing_pop = pop_diffs.loc[country, f"pop_{level}_{year}"]

                probs = temp.query(f"NAME_0 == '{country}'")[f"pop_{level}_{year}"]
                probs = probs / probs.sum()
                temp.loc[temp["NAME_0"] == country, f"pop_{level}_{year}"] -= (
                    probs * missing_pop
                )

    for year in range(1975, YEAR + 1, 5):
        temp = temp.assign(
            **{
                f"pop_{year}": lambda df: (
                    df[f"pop_rural_{year}"]
                    + df[f"pop_urban_center_{year}"]
                    + df[f"pop_urban_cluster_{year}"]
                ),
                f"density_{year}": lambda df: df[f"pop_{year}"] / df["area_km2"],
            },
        )

    temp = temp.sort_values([f"NAME_{i}" for i in range(5)] + ["name"])

    pop_diffs = temp.groupby("NAME_0").agg(
        {
            f"pop_{category}_{year}": "sum"
            for year in range(1975, 2021, 5)
            for category in ["rural", "urban_center", "urban_cluster"]
        },
    ) - df_correct.groupby("NAME_0").agg(
        {
            f"pop_{category}_{year}": "sum"
            for year in range(1975, 2021, 5)
            for category in ["rural", "urban_center", "urban_cluster"]
        },
    )
    pop_diffs = pop_diffs[~pop_diffs.isna().all(axis=1)]

    temp = temp.join(df_split[["area_rural_km2", "area_urban_cluster_km2", "area_urban_center_km2"]])

In [190]:
split_path = sent_path / "DEGURBA" / "polígonos" / str(YEAR) / "sin_conurb.gpkg"
split_path.parent.mkdir(parents=True, exist_ok=True)

df_split.to_file(split_path)

In [192]:
# df_final.drop(columns=["geometry"]).to_excel(
#     sent_path / "DEGURBA" / "hojas" / "normal.xlsx",
#     index=False,
# )

if YEAR == 2020:
    temp.to_excel(
        sent_path / "DEGURBA" / "hojas" / str(YEAR) / "sin_conurb.xlsx",
        index=False,
    )

    # temp.groupby("NAME_0")[
    #     [
    #         f"pop_{YEAR}",
    #         f"pop_urban_center_{YEAR}",
    #         f"pop_urban_cluster_{YEAR}",
    #         f"pop_rural_{YEAR}",
    #         "area_km2",
    #     ]
    # ].sum().round(1).to_excel(stats_path / f"{YEAR}.xlsx")

else:
    split_sheet_path = sent_path / "DEGURBA" / "hojas" / str(YEAR) / "sin_conurb.xlsx"
    split_sheet_path.parent.mkdir(parents=True, exist_ok=True)

    df_split.drop(columns=["geometry"]).to_excel(
        split_sheet_path,
        index=False,
    )

    # df_split.groupby("NAME_0")[
    #     [
    #         f"pop_{YEAR}",
    #         f"pop_urban_center_{YEAR}",
    #         f"pop_urban_cluster_{YEAR}",
    #         f"pop_rural_{YEAR}",
    #         "area_km2",
    #     ]
    # ].sum().round(1).to_excel(stats_path / f"{YEAR}.xlsx")

In [57]:
def generate_stats_df(df: pd.DataFrame) -> pd.DataFrame:
    return (
        df.groupby("NAME_0")[
            [
                f"pop_{YEAR}",
                f"pop_urban_center_{YEAR}",
                f"pop_urban_cluster_{YEAR}",
                f"pop_rural_{YEAR}",
                "area_km2",
            ]
        ]
        .sum()
        .round(1)
    )

In [58]:
temp["area"]

NameError: name 'temp' is not defined

In [ ]:
generate_stats_df(temp)

,pop_2020,pop_urban_center_2020,pop_urban_cluster_2020,pop_rural_2020,area_km2
NAME_0,,,,,
Argentina,41204606.9,28861957.5,9875914.6,2466734.9,17218.0
Bahamas,360525.6,252204.9,82802.1,25518.6,379.0
Barbados,263432.0,152414.5,89967.0,21050.5,298.0
Belice,296095.0,72849.0,148646.0,74600.0,360.0
Bolivia,9877516.7,6623655.7,1935199.1,1318661.9,5736.0
Brasil,186254104.8,116654597.0,52697976.9,16901530.9,86957.0
Chile,16976510.9,12611093.6,3169247.0,1196170.2,7603.0
Colombia,45167624.2,31777291.4,9666193.1,3724139.8,15440.0
Costa Rica,4310938.0,2515672.0,1275448.0,519818.0,3144.0
